# **Approach 4:**

In [2]:
# # installations if needed
# !pip install -qq torchvision
# !pip install -qq hdbscan
# !pip install -qq rapidfuzz

In [3]:
# Import libraries
from IPython.display import display, HTML
from PIL import Image
from collections import Counter, defaultdict
from io import BytesIO
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import pair_confusion_matrix, silhouette_score
from sklearn.metrics.cluster import pair_confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances
from sklearn.preprocessing import OneHotEncoder, normalize
from tqdm import tqdm
import hdbscan
import itertools
import json
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import requests
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from sklearn.cluster import KMeans
from itertools import count
from sklearn.cluster import DBSCAN

### Function to visualize products images

In [5]:
# function to visualise these clusters as HTML
def show_clusters_as_html(df, cluster_ids):
    for cluster_id in cluster_ids:
        cluster_df = df[df['cluster_label'] == cluster_id]
        if cluster_df.empty:
            print(f"Cluster {cluster_id} is empty!")
            continue

        html = f"<h3 style='margin-top:20px'>Cluster {cluster_id} ({len(cluster_df)} items)</h3>"
        html += "<div style='display:flex;flex-wrap:wrap'>"

        for _, row in cluster_df.iterrows():
            name = row.get('product_name', '')
            
            # Choose the best available image
            for col in [
                'mainImage.original_url',
                'secondary_image_1.original_url',
                'secondary_image_2.original_url',
                'secondary_image_3.original_url'
            ]:
                img = row.get(col, '')
                if isinstance(img, str) and img.startswith('http'):
                    break
            else:
                img = ''
            
            html += f"""
                <div style='margin:10px;text-align:center'>
                    <img src='{img}' width='120' style='border-radius:8px'><br>
                    <div style='max-width:120px;font-size:12px'>{name}</div>
                </div>
            """
        html += "</div>"
        display(HTML(html))

# **EDA & preprocessing**

In [8]:
# Read data
df = pd.read_json('sneakers_sample_19062025.txt', lines=False)

print(df.head())

                                                data  \
0  {'EAN': [], 'size': '44', 'brand': 'ASICS', 'c...   
1  {'EAN': [], 'size': '44', 'brand': 'VANS', 'co...   
2  {'EAN': [], 'size': '44.5', 'brand': 'ADIDAS',...   
3  {'EAN': [], 'size': '42', 'brand': 'NIKE', 'co...   
4  {'EAN': [], 'size': '11/EU 46', 'brand': 'PUMA...   

                      mirakl_product_id             creation_date  \
0  d96975c5-0bea-41f6-a4a8-fdac53f374f3  2024-01-09T12:09:20.568Z   
1  6f302b78-9a86-484e-894f-bb39220931a3  2023-11-09T13:08:33.276Z   
2  70006973-aa9f-4176-9f38-0ba2c5a1b06f  2025-03-29T08:43:47.421Z   
3  6b010b1d-7475-44dc-ad56-502769d2e742  2024-01-04T12:29:59.343Z   
4  cf04d217-bfd6-4743-ae4b-bc98712959ca  2025-02-13T14:31:57.524Z   

                update_date                           product_sku  \
0  2024-11-06T13:03:01.469Z  d96975c5-0bea-41f6-a4a8-fdac53f374f3   
1  2025-02-12T12:21:44.083Z  6f302b78-9a86-484e-894f-bb39220931a3   
2  2025-03-30T21:07:23.464Z  70006973-aa9

In [9]:
df.head()

,data,mirakl_product_id,creation_date,update_date,product_sku,validation,synchronization,catalogs,product_urls,sources,selling_authorization,data_origin
0,"{'EAN': [], 'size': '44', 'brand': 'ASICS', 'c...",d96975c5-0bea-41f6-a4a8-fdac53f374f3,2024-01-09T12:09:20.568Z,2024-11-06T13:03:01.469Z,d96975c5-0bea-41f6-a4a8-fdac53f374f3,"{'status': 'VALID', 'validation_errors': []}",{'status': 'SYNCHRONIZED'},[],[],"[{'provider_code': '2077', 'provider_sku': '10...","{'restricted': False, 'authorized_selling_shop...","{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."
1,"{'EAN': [], 'size': '44', 'brand': 'VANS', 'co...",6f302b78-9a86-484e-894f-bb39220931a3,2023-11-09T13:08:33.276Z,2025-02-12T12:21:44.083Z,6f302b78-9a86-484e-894f-bb39220931a3,"{'status': 'VALID', 'validation_errors': []}",{'status': 'SYNCHRONIZED'},[],[],"[{'provider_code': '2002', 'provider_sku': 'VN...","{'restricted': False, 'authorized_selling_shop...","{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."
2,"{'EAN': [], 'size': '44.5', 'brand': 'ADIDAS',...",70006973-aa9f-4176-9f38-0ba2c5a1b06f,2025-03-29T08:43:47.421Z,2025-03-30T21:07:23.464Z,70006973-aa9f-4176-9f38-0ba2c5a1b06f,"{'status': 'VALID', 'validation_errors': []}",{'status': 'SYNCHRONIZED'},[],[],"[{'provider_code': '2167', 'provider_sku': 'IH...","{'restricted': False, 'authorized_selling_shop...","{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."
3,"{'EAN': [], 'size': '42', 'brand': 'NIKE', 'co...",6b010b1d-7475-44dc-ad56-502769d2e742,2024-01-04T12:29:59.343Z,2024-11-06T13:00:53.470Z,6b010b1d-7475-44dc-ad56-502769d2e742,"{'status': 'VALID', 'validation_errors': []}",{'status': 'SYNCHRONIZED'},[],[],"[{'provider_code': '2077', 'provider_sku': 'CT...","{'restricted': False, 'authorized_selling_shop...","{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."
4,"{'EAN': [], 'size': '11/EU 46', 'brand': 'PUMA...",cf04d217-bfd6-4743-ae4b-bc98712959ca,2025-02-13T14:31:57.524Z,2025-03-01T00:03:24.205Z,cf04d217-bfd6-4743-ae4b-bc98712959ca,"{'status': 'VALID', 'validation_errors': []}",{'status': 'SYNCHRONIZED'},[],[],"[{'provider_code': '3390', 'provider_sku': '46...","{'restricted': False, 'authorized_selling_shop...","{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."


### Create separate dataframes

In [11]:
# Load the JSON file
with open('sneakers_sample_19062025.txt', 'r', encoding='utf-8') as f:    # added encoding = utf8 otherwise: decoding error
    raw_data = json.load(f)

# Create separate dataframes
meta_rows = []
data_rows = []
sources_rows = []

# Exclude some fields that are not necessary for our tasks
exclude_fields = {
    'creation_date', 'update_date',
    'validation', 'synchronization',
    'catalogs', 'selling_authorization', "product_urls"
}

for entry in raw_data:
    # Top-level fields
    meta = {
        key: entry[key] for key in entry
        if key not in exclude_fields and key not in ['data', 'sources']
    }
    meta_rows.append(meta)
    
    # Extract nested fields (if they exist)
    if 'data' in entry:
        data_rows.append(entry['data'])
    if 'sources' in entry and isinstance(entry['sources'], list):
        for source in entry['sources']:
            # Optionally, attach product ID to track back
            source['mirakl_product_id'] = entry.get('mirakl_product_id')
            sources_rows.append(source)

# Convert all parts to DataFrames
df_meta = pd.DataFrame(meta_rows)
df_data = pd.json_normalize(data_rows)
df_sources = pd.DataFrame(sources_rows)

# Preview
print("Meta columns:", df_meta.columns.tolist())
print("Data columns:", df_data.columns.tolist())
print("Sources sample:", df_sources.head())

Meta columns: ['mirakl_product_id', 'product_sku', 'data_origin']
Data columns: ['EAN', 'size', 'brand', 'color', 'category', 'returnable', 'name [el_CY]', 'name [en_GB]', 'updateOnDelta', 'refinementColor', 'variantGroupCode', 'jiniusSkuMatchCode', 'shortDescription [el_CY]', 'shortDescription [en_GB]', 'mainImage.source', 'mainImage.original_url', 'secondary_image_1.source', 'secondary_image_1.original_url', 'secondary_image_2.source', 'secondary_image_2.original_url', 'secondary_image_3.source', 'secondary_image_3.original_url', 'longDescription [en_GB]', 'longDescription [el_CY]', 'shoeType', 'shoeShape', 'fashionMaterial', 'shoeWidth', 'heelHeight', 'name', 'heelType', 'longDescription', 'shortDescription']
Sources sample:   provider_code                               provider_sku  \
0          2077        1071A091-101--EUM-18--WHITE / BLACK   
1          2002                   VN0A3MTJT2J1--417--BLACK   
2          2167                                IH6003-44.5   
3          207

In [12]:
df_data

,EAN,size,brand,color,category,returnable,name [el_CY],name [en_GB],updateOnDelta,refinementColor,...,longDescription [el_CY],shoeType,shoeShape,fashionMaterial,shoeWidth,heelHeight,name,heelType,longDescription,shortDescription
0,[],44,ASICS,White / Black,FaMeShSneakers,true,Gel-rocket 11 mens volleyball/indoor shoes,Gel-rocket 11 mens volleyball/indoor shoes,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,[],44,VANS,Black,FaMeShSneakers,true,Mens filmore suede,Mens filmore suede,None,black,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,[],44.5,ADIDAS,NaN,FaMeShSneakers,true,Ih6003,Adidas response,NaN,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,[],42,NIKE,White / Black / Volt,FaMeShSneakers,true,React vision worldwide mens,React vision worldwide mens,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,[],11/EU 46,PUMA,Black/white,FaMeShSneakers,true,Puma men's x-ray 3 sd,Puma men's x-ray 3 sd,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5818,[],37,BUFFALO,White / Silver / Black,FaWoShSneakers,true,Triple hollow sneakers,Triple hollow sneakers,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5819,[],39,ADIDAS ORIGINALS,White,FaWoShSneakers,true,Adidas sambae w,Adidas sambae w,NaN,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5820,[],41,GUESS,White,FaWoShSneakers,true,Elbina sneakers,Elbina sneakers,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5821,[],36.5,NEW BALANCE,Cloud White / Burgundy,FaWoShSneakers,true,327 sportstyle sneakers,327 sportstyle sneakers,None,miscellaneous,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
df_meta

,mirakl_product_id,product_sku,data_origin
0,d96975c5-0bea-41f6-a4a8-fdac53f374f3,d96975c5-0bea-41f6-a4a8-fdac53f374f3,"{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."
1,6f302b78-9a86-484e-894f-bb39220931a3,6f302b78-9a86-484e-894f-bb39220931a3,"{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."
2,70006973-aa9f-4176-9f38-0ba2c5a1b06f,70006973-aa9f-4176-9f38-0ba2c5a1b06f,"{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."
3,6b010b1d-7475-44dc-ad56-502769d2e742,6b010b1d-7475-44dc-ad56-502769d2e742,"{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."
4,cf04d217-bfd6-4743-ae4b-bc98712959ca,cf04d217-bfd6-4743-ae4b-bc98712959ca,"{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."
...,...,...,...
5818,acdb8093-c93c-49d7-af05-f04cae009f99,acdb8093-c93c-49d7-af05-f04cae009f99,"{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."
5819,e733c608-da84-49f1-a3ae-52a025b1cb3d,e733c608-da84-49f1-a3ae-52a025b1cb3d,"{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."
5820,6d8cf85e-93bd-47ea-8a59-3e43fe4ef732,6d8cf85e-93bd-47ea-8a59-3e43fe4ef732,"{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."
5821,e2d00db3-27b2-4996-89e2-5c1579af5d39,e2d00db3-27b2-4996-89e2-5c1579af5d39,"{'name [en_GB]': {'origin': 'PROVIDER', 'provi..."


We observe that `data_origin` column is nested.

In [15]:
# Flatten 'data_origin' column
data_origin_df = pd.json_normalize(df_meta['data_origin'])

# Add prefixes so we know where they came from
data_origin_df = data_origin_df.add_prefix('origin_')

# Drop original nested columns from df_meta
df_meta = df_meta.drop(columns=['data_origin'])

# Merge the flattened parts back
df_meta = pd.concat([df_meta, data_origin_df], axis=1)

In [16]:
df_meta

,mirakl_product_id,product_sku,origin_name [en_GB].origin,origin_name [en_GB].provider_code,origin_name [el_CY].origin,origin_name [el_CY].provider_code,origin_color.origin,origin_color.provider_code,origin_shortDescription [el_CY].origin,origin_shortDescription [el_CY].provider_code,...,origin_heelHeight.origin,origin_heelHeight.provider_code,origin_longDescription.origin,origin_longDescription.provider_code,origin_shortDescription.origin,origin_shortDescription.provider_code,origin_name.origin,origin_name.provider_code,origin_heelType.origin,origin_heelType.provider_code
0,d96975c5-0bea-41f6-a4a8-fdac53f374f3,d96975c5-0bea-41f6-a4a8-fdac53f374f3,PROVIDER,2077,PROVIDER,2077,PROVIDER,2077,PROVIDER,2077,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,6f302b78-9a86-484e-894f-bb39220931a3,6f302b78-9a86-484e-894f-bb39220931a3,PROVIDER,2002,PROVIDER,2002,PROVIDER,2002,PROVIDER,2002,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,70006973-aa9f-4176-9f38-0ba2c5a1b06f,70006973-aa9f-4176-9f38-0ba2c5a1b06f,PROVIDER,2167,PROVIDER,2167,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,6b010b1d-7475-44dc-ad56-502769d2e742,6b010b1d-7475-44dc-ad56-502769d2e742,PROVIDER,2077,PROVIDER,2077,PROVIDER,2077,PROVIDER,2077,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,cf04d217-bfd6-4743-ae4b-bc98712959ca,cf04d217-bfd6-4743-ae4b-bc98712959ca,PROVIDER,3390,PROVIDER,3390,PROVIDER,3390,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5818,acdb8093-c93c-49d7-af05-f04cae009f99,acdb8093-c93c-49d7-af05-f04cae009f99,PROVIDER,2077,PROVIDER,2077,PROVIDER,2077,PROVIDER,2077,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5819,e733c608-da84-49f1-a3ae-52a025b1cb3d,e733c608-da84-49f1-a3ae-52a025b1cb3d,PROVIDER,3429,PROVIDER,3429,PROVIDER,3429,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5820,6d8cf85e-93bd-47ea-8a59-3e43fe4ef732,6d8cf85e-93bd-47ea-8a59-3e43fe4ef732,PROVIDER,2002,PROVIDER,2002,PROVIDER,2002,PROVIDER,2002,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5821,e2d00db3-27b2-4996-89e2-5c1579af5d39,e2d00db3-27b2-4996-89e2-5c1579af5d39,PROVIDER,2077,PROVIDER,2077,PROVIDER,2077,PROVIDER,2077,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
df_sources

,provider_code,provider_sku,mirakl_product_id
0,2077,1071A091-101--EUM-18--WHITE / BLACK,d96975c5-0bea-41f6-a4a8-fdac53f374f3
1,2002,VN0A3MTJT2J1--417--BLACK,6f302b78-9a86-484e-894f-bb39220931a3
2,2167,IH6003-44.5,70006973-aa9f-4176-9f38-0ba2c5a1b06f
3,2077,CT2927-100--USAM-09--WHITE / BLACK / VOLT,6b010b1d-7475-44dc-ad56-502769d2e742
4,3390,468781-1870785842,cf04d217-bfd6-4743-ae4b-bc98712959ca
...,...,...,...
5818,2077,1636155--EUM-05--WHITE / SILVER / BLACK,acdb8093-c93c-49d7-af05-f04cae009f99
5819,3429,JI1349--1310--WHITE,e733c608-da84-49f1-a3ae-52a025b1cb3d
5820,2002,FLJELBLEA12WHBEI--401--WHITE,6d8cf85e-93bd-47ea-8a59-3e43fe4ef732
5821,2077,WS327-KA--EUM-04--CLOUD WHITE / BURGUNDY,e2d00db3-27b2-4996-89e2-5c1579af5d39


Our algorithms are **evaluated** on a manually labeled sample dataset. The labels correspond to variant groupings, which means that **only Task 2 can currently be evaluated**. However, Task 1 could be evaluated in a similar manner, provided we have a sample dataset labeled specifically for that task.

In [19]:
# Load the sample to evaluate the groups
sample_df = pd.read_csv("sample.csv")

Now that we have separate flattened dataframes we can investigate whether we have missing values and in which fields.

In [22]:
df_data.isna().sum()

EAN                                  0
size                                 0
brand                               11
color                               15
category                             0
returnable                           0
name [el_CY]                       608
name [en_GB]                         0
updateOnDelta                     5819
refinementColor                    733
variantGroupCode                     0
jiniusSkuMatchCode                   0
shortDescription [el_CY]          1336
shortDescription [en_GB]             0
mainImage.source                     0
mainImage.original_url               4
secondary_image_1.source           290
secondary_image_1.original_url     292
secondary_image_2.source           437
secondary_image_2.original_url     437
secondary_image_3.source           737
secondary_image_3.original_url     737
longDescription [en_GB]           5585
longDescription [el_CY]           5716
shoeType                          5681
shoeShape                

In [23]:
df_meta.isna().sum()

mirakl_product_id                                   0
product_sku                                         0
origin_name [en_GB].origin                          0
origin_name [en_GB].provider_code                 153
origin_name [el_CY].origin                        608
origin_name [el_CY].provider_code                 682
origin_color.origin                                15
origin_color.provider_code                        155
origin_shortDescription [el_CY].origin           1336
origin_shortDescription [el_CY].provider_code    1340
origin_secondary_image_2.origin                   437
origin_secondary_image_2.provider_code            446
origin_secondary_image_1.origin                   288
origin_secondary_image_1.provider_code            306
origin_shortDescription [en_GB].origin              0
origin_shortDescription [en_GB].provider_code      61
origin_secondary_image_3.origin                   737
origin_secondary_image_3.provider_code            741
origin_mainImage.origin     

In [24]:
df_sources.isna().sum()

provider_code        0
provider_sku         0
mirakl_product_id    0
dtype: int64

From the above we observe that some important fields (such as color, brand, descriptions) contain missing values.

In [27]:
# check the values of jiniusSkuMatchCode column
df_data["jiniusSkuMatchCode"].describe() 

count     5823
unique       1
top         []
freq      5823
Name: jiniusSkuMatchCode, dtype: object

`jiniusSkuMatchCode` column is only has empty lists, although it appeared to have no missing values.

In [29]:
#check the number of records with empty EAN
df_data["EAN"].describe()

count     5823
unique     487
top         []
freq      5337
Name: EAN, dtype: object

In [30]:
df_data["EAN"].apply(lambda x: x == []).sum()

5337

From the 5823 observations, 5337 of them have empty EAN code and the remaining have a unique EAN code. So no products are grouped by the EAN code.

In [32]:
df_data['brand'].isna().sum()

11

In [33]:
df_data[df_data['brand'].isna()][['name [en_GB]','name', 'color', 'category', 'shortDescription', 'brand']]

,name [en_GB],name,color,category,shortDescription,brand
1075,C8979,NaN,Black,FaMeShSneakers,NaN,NaN
2258,C8979,NaN,All White,FaMeShSneakers,NaN,NaN
2559,C8979,NaN,Black,FaMeShSneakers,NaN,NaN
2941,C895,NaN,White,FaWoShSneakers,NaN,NaN
3856,C895,NaN,Black,FaWoShSneakers,NaN,NaN
4193,C667,NaN,White,FaWoShSneakers,NaN,NaN
4469,C872,NaN,White,FaWoShSneakers,NaN,NaN
4777,C8962,NaN,Black,FaWoShSneakers,NaN,NaN
5033,C667,NaN,White,FaWoShSneakers,NaN,NaN
5451,C666,NaN,White,FaWoShSneakers,NaN,NaN


In [34]:
df_data[df_data['brand'] == 'CALVIN KLEIN JEANS']

,EAN,size,brand,color,category,returnable,name [el_CY],name [en_GB],updateOnDelta,refinementColor,...,longDescription [el_CY],shoeType,shoeShape,fashionMaterial,shoeWidth,heelHeight,name,heelType,longDescription,shortDescription
148,[],45,CALVIN KLEIN JEANS,Black / White,FaMeShSneakers,true,Basket cupsole low leather sneakers,Basket cupsole low leather sneakers,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
335,[],45,CALVIN KLEIN JEANS,White,FaMeShSneakers,true,Retro tennis low laceup mtl,Retro tennis low laceup mtl,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
660,[],45,CALVIN KLEIN JEANS,White,FaMeShSneakers,true,Senakers,Senakers,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
675,[],42,CALVIN KLEIN JEANS,Black,FaMeShSneakers,true,Chunky cupsole authentic shoe,Chunky cupsole authentic shoe,None,black,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1204,[],43,CALVIN KLEIN JEANS,White,FaMeShSneakers,true,Chunky cupsole mono leather,Chunky cupsole mono leather,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1297,[],44,CALVIN KLEIN JEANS,Black,FaMeShSneakers,true,Retro tennis low laceup mtl,Retro tennis low laceup mtl,None,black,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1992,[],41,CALVIN KLEIN JEANS,White,FaMeShSneakers,true,Claic cupsole iconic nylon shoe,Claic cupsole iconic nylon shoe,NaN,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3580,[],40,CALVIN KLEIN JEANS,Black,FaWoShSneakers,true,Chunky cupsole laceup leather shoe,Chunky cupsole laceup leather shoe,None,black,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5088,[],40,CALVIN KLEIN JEANS,White,FaWoShSneakers,true,Retro tennis su-mesh wn,Retro tennis su-mesh wn,None,white,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
df_data[df_data['brand'] == 'CALVIN KLEIN']

,EAN,size,brand,color,category,returnable,name [el_CY],name [en_GB],updateOnDelta,refinementColor,...,longDescription [el_CY],shoeType,shoeShape,fashionMaterial,shoeWidth,heelHeight,name,heelType,longDescription,shortDescription
4411,[5299901986314],36,CALVIN KLEIN,White,FaWoShSneakers,true,Eva runner low lace shoes,Eva runner low lace shoes,None,white,...,NaN,0090,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The outputs above reveal inconsistencies in brand naming (e.g., *LIU JO SHOES* vs *LIU JO*). To ensure consistency, we apply a brand name mapping that standardizes all variations.

In [37]:
df_data['brand'].unique()

array(['ASICS', 'VANS', 'ADIDAS', 'NIKE', 'PUMA', 'CONVERSE',
       'ADIDAS ORIGINALS', 'UNDER ARMOUR', 'SAUCONY', 'REEBOK', 'JORDAN',
       'DIESEL', 'CHAMPION', 'NEW BALANCE', 'FRED PERRY', 'SKECHERS',
       'ON RUNNING', 'EA7', 'JOMA', 'CARRERA', "LEVI'S", 'VEJA',
       'PEPE SHOES', 'TALBOT SHOES', 'U.S. POLO ASSN.',
       'PEPE JEANS LONDON', 'TOMMY HILFIGER', 'CALVIN KLEIN JEANS', 'DC',
       'STEFAN', 'ANTONY MORATO', 'TOMMY JEANS', 'KAPPA', 'LYLE & SCOTT',
       'HACKETT', 'KARL KANI', 'SERGIO TACCHINI', 'TAMARIS', 'REVERSE',
       'BAZAAR CHARM', 'MORK EXCLUSIVE', 'LACOSTE', 'SOUTHPORT',
       'SALOMON', 'GAUDI', 'JACK & JONES', 'FUNKY BUDDHA', 'HUGO BOSS',
       'FENOMILANO', 'FILA', 'RHAPSODY', 'BOSTON SPORT CLUB', 'MIMSOGA',
       'SUN68', 'LIU JO SHOES', 'LOTTO', 'PLATO', 'MIZUNO', 'ADMIRAL',
       'BROOKS', 'SIKSILK', 'NICO', 'ICE PLAY', 'BHPC', 'GUESS DENIM',
       'PARIS', 'NO NAME ATHENS', 'GUESS', 'ATLANTA', nan,
       'BE YOU, U.S. POLO ASSN', 'THE NORT

In [38]:
brand_mapping = {
    'ADIDAS ORIGINALS': 'ADIDAS',
    'CALVIN KLEIN JEANS': 'CALVIN KLEIN',
    'LIU JO SHOES': 'LIU JO',
    'GUESS DENIM': 'GUESS',
    'TOMMY JEANS': 'TOMMY HILFIGER',
    'PEPE JEANS LONDON': 'PEPE JEANS',
    'BE YOU, U.S. POLO ASSN': 'U.S. POLO ASSN.',
    'BHPC': 'BEVERLY HILLS POLO CLUB',
    'EA7': 'EMPORIO ARMANI',
    'NO NAME ATHENS': 'NO NAME'
}

In [39]:
df_data_copy = df_data.copy()

df_data_copy['brand_label'] = df_data_copy['brand'].replace(brand_mapping)

In [40]:
df_data['brand'].nunique()

128

In [41]:
df_data_copy['brand'].nunique()

128

In [42]:
df_data_copy['brand_label'].nunique()

121

In [43]:
df_data_copy[df_data_copy['brand_label'] == 'ADIDAS']['name [en_GB]'].unique()

array(['Adidas response', 'Adidas gazelle shoes', 'Swift run 22',
       'Adistar 2 men', 'Midcity mid',
       "Adidas men's supernova stride shoes", 'Duramo speed men',
       'Adidas gazelle indoor', 'Gazelle shoes', 'Cloudfoam go lounge',
       'Midcity low', 'Strutter', 'Response runner unisex',
       'Response super mens', 'Adidas grand court base 2.0',
       'Stan smith vegan', 'Forest grove sneakers', 'Znsored hi mens',
       'Fukasa run', 'Cloudfoam comfy', 'Urban court',
       'Adidas alphaboost v1', 'Alphabounce + c', 'Samba classic',
       'Copa mundial', 'Ozelle', 'Adistar 3', 'X_plrphase',
       'Ultimashow 2.0',
       'Adidas questar             greone/cblack/grethr', 'Advantage 2.0',
       'Adidas copa pure.3 tf', 'Grand court 2.0', 'Samba og',
       'Adidas handball spezial', 'Handball spezial', 'Runfalcon 3.0',
       'Runfalcon 5', 'Adidas men duramo sl', 'Copa pure.1 firm ground',
       "Adidas men's duramo speed shoes", 'Park st',
       'Adidas men runf

In [47]:
# inspect features
print(df_data.columns)

Index(['EAN', 'size', 'brand', 'color', 'category', 'returnable',
       'name [el_CY]', 'name [en_GB]', 'updateOnDelta', 'refinementColor',
       'variantGroupCode', 'jiniusSkuMatchCode', 'shortDescription [el_CY]',
       'shortDescription [en_GB]', 'mainImage.source',
       'mainImage.original_url', 'secondary_image_1.source',
       'secondary_image_1.original_url', 'secondary_image_2.source',
       'secondary_image_2.original_url', 'secondary_image_3.source',
       'secondary_image_3.original_url', 'longDescription [en_GB]',
       'longDescription [el_CY]', 'shoeType', 'shoeShape', 'fashionMaterial',
       'shoeWidth', 'heelHeight', 'name', 'heelType', 'longDescription',
       'shortDescription'],
      dtype='object')


In [48]:
#column "name" is empty in almost all rows
df_data["name"].isna().sum()/df_data.shape[0]

0.9958784131890778

In [49]:
#find the only products with non-empty "name" column
df_data[df_data["name"].notna()]

,EAN,size,brand,color,category,returnable,name [el_CY],name [en_GB],updateOnDelta,refinementColor,...,longDescription [el_CY],shoeType,shoeShape,fashionMaterial,shoeWidth,heelHeight,name,heelType,longDescription,shortDescription
1340,[198376576226],43,SKECHERS,Black,FaMeShSneakers,true,Skechers 118317 men shoes,Skechers 118317 men shoes,NaN,black,...,SKECHERS 118317 MEN SHOES / GLIDE STEP,0030,NaN,NaN,medium,NaN,SKECHERS 118317 MEN SHOES / GLIDE STEP,other,SKECHERS 118317 MEN SHOES / GLIDE STEP,SKECHERS 118317 MEN SHOES
1458,[198376147365],41,SKECHERS,Black,FaMeShSneakers,true,210890 bbk skechers men shoes,210890 bbk skechers men shoes,NaN,black,...,210890 BBK SKECHERS MEN SHOES/ SLIP IN,0030,NaN,NaN,medium,NaN,210890 BBK SKECHERS MEN SHOES,other,210890 BBK SKECHERS MEN SHOES/ SLIP IN,210890 BBK SKECHERS MEN SHOES
1805,[196989378503],42,SKECHERS,Grey,FaMeShSneakers,true,Skechers 232700 gry men shoes,Skechers 232700 grymen shoes,NaN,grey,...,SKECHERS 232700 GRY MEN SHOES / ARCH FIT,0020,NaN,NaN,medium,NaN,SKECHERS 232700 GRY MEN SHOES / ARCH FIT,other,SKECHERS 232700 GRY MEN SHOES / ARCH FIT,SKECHERS 232700 GRY MEN SHOES
1853,[196989378527],43,SKECHERS,Grey,FaMeShSneakers,true,Skechers 232700 gry men shoes,Skechers 232700 grymen shoes,NaN,grey,...,SKECHERS 232700 GRY MEN SHOES / ARCH FIT,0020,NaN,NaN,medium,NaN,SKECHERS 232700 GRY MEN SHOES / ARCH FIT,other,SKECHERS 232700 GRY MEN SHOES / ARCH FIT,SKECHERS 232700 GRY MEN SHOES
2443,[198376147419],43,SKECHERS,Black,FaMeShSneakers,true,210890 bbk skechers men shoes,210890 bbk skechers men shoes,NaN,black,...,210890 BBK SKECHERS MEN SHOES/ SLIP IN,0030,NaN,NaN,medium,NaN,210890 BBK SKECHERS MEN SHOES,other,210890 BBK SKECHERS MEN SHOES/ SLIP IN,210890 BBK SKECHERS MEN SHOES
3112,[197976864054],41,SKECHERS,Black,FaWoShSneakers,true,Skechers 150370 bbk women shoes,Skechers 150370 bbk women shoes,None,black,...,SKECHERS 150370 BBK WOMEN SHOES / AIR SOLE,0020,NaN,NaN,medium,NaN,SKECHERS 150370 BBK WOMEN SHOES,other,SKECHERS 150370 BBK WOMEN SHOES / AIR SOLE,SKECHERS 150370 BBK WOMEN SHOES
3455,[196989863429],38,SKECHERS,Black/gold,FaWoShSneakers,true,Skechers 117513 women shoes,Skechers 117513 women shoes,None,black,...,SKECHERS 117513 WOMEN SHOES / MEMORY FOAM,0020,NaN,NaN,medium,NaN,SKECHERS 117513 WOMEN SHOES / BOBS SPORT,other,SKECHERS 117513 WOMEN SHOES / MEMORY FOAM,SKECHERS 117513 WOMEN SHOES
3457,[196989863405],37,SKECHERS,Black/gold,FaWoShSneakers,true,Skechers 117513 women shoes,Skechers 117513 women shoes,None,black,...,SKECHERS 117513 WOMEN SHOES / MEMORY FOAM,0020,NaN,NaN,medium,NaN,SKECHERS 117513 WOMEN SHOES / BOBS SPORT,other,SKECHERS 117513 WOMEN SHOES / MEMORY FOAM,SKECHERS 117513 WOMEN SHOES
3498,[198376755850],39,SKECHERS,Beige/silver,FaWoShSneakers,true,Skechers 150246 ntsl women shoes,Skechers 150246 ntsl women shoes,NaN,beige,...,SKECHERS 150246 NTSL WOMEN SHOES / MEMORY FOAM,0020,NaN,NaN,medium,NaN,SKECHERS 150246 NTSL WOMEN SHOES,other,SKECHERS 150246 NTSL WOMEN SHOES / MEMORY FOAM,SKECHERS 150246 NTSL WOMEN SHOES
3550,[198739014495],41,SKECHERS,Black,FaWoShSneakers,true,Skechers 150562 bklv women shoes,Skechers 150562 bklv women shoes,NaN,black,...,SKECHERS 150562 BKLV WOMEN SHOES / MEMORY FOAM,0020,NaN,NaN,medium,NaN,SKECHERS 150562 BKLV WOMEN SHOES,other,SKECHERS 150562 BKLV WOMEN SHOES / MEMORY FOAM,SKECHERS 150562 BKLV WOMEN SHOES


In [50]:
#check if some name [el_CY] are missing
df_data["name [el_CY]"].isna().sum()

608

In [51]:
#check if some name [en_GB] are missing
df_data["name [en_GB]"].isna().sum() #no missing values

0

Given that `name [el_CY]` has missing values in some rows, while `name [en_GB]` not, it is better not to use the Greek name in the following grouping to provide more accurate results. 

A similar check for English and Greek descriptions.

In [53]:
#check if some shortDescription [el_CY] are missing
df_data["shortDescription [el_CY]"].isna().sum() 

1336

In [54]:
#check if some shortDescription [en_GB] are missing
df_data["shortDescription [en_GB]"].isna().sum() #no missing values

0

Again, some Greek descriptions are missing, while there are no missing English descriptions. So it would be better to use only English.

## Approach 4

## Attempting contextual embeddings for metadata

Using TFIDF may be suboptimal as "Nike Air Max" ≠ "Nike Airmax", "White and Black" ≠ "Black/White". 

However, Sentence-BERT understands the meaning of the full sentence. So we may use this transformer to encode the text metadata.

In [69]:
# === Step 1: Load Data and Image Embeddings ===
df = pd.read_json('sneakers_sample_19062025.txt', lines=False)
image_vecs = np.load("image_features.npy")

# Align DataFrame to match saved image embeddings
df = df.reset_index(drop=True).iloc[:len(image_vecs)]

# === Step 2: Extract Text Metadata ===
df["description"] = df["data"].apply(lambda x: x.get("shortDescription") if isinstance(x, dict) else "")
df["name"] = df["data"].apply(lambda x: x.get("name [en_GB]") if isinstance(x, dict) else "")
df["brand"] = df["data"].apply(lambda x: x.get("brand") if isinstance(x, dict) else "")
df["color"] = df["data"].apply(lambda x: x.get("color") if isinstance(x, dict) else "")

# === Step 3: Sentence-BERT for Text Embeddings ===
text_model = SentenceTransformer("all-MiniLM-L6-v2")
texts = (df["name"].fillna("") + " " + df["brand"].fillna("") + " " + df["color"].fillna("") + " " + df["description"].fillna("")).tolist()
text_vecs = text_model.encode(texts, normalize_embeddings=True)

# === Step 4: Combine Vectors by Concatenation ===
image_vecs = normalize(image_vecs)
alpha = 0.6  # weight emphasis for image vs text
combined_vecs = np.hstack([alpha * image_vecs, (1 - alpha) * text_vecs])
combined_vecs = normalize(combined_vecs)

# === Step 5: HDBSCAN Clustering ===
clusterer = hdbscan.HDBSCAN(min_cluster_size=2, metric='euclidean')
df["cluster_id"] = clusterer.fit_predict(combined_vecs)

# === Step 6: Export Cluster Groups to JSON ===
groups = defaultdict(list)
for _, row in df.iterrows():
    groups[int(row["cluster_id"])].append({
        "product_id": row.get("mirakl_product_id"),
        "name": row.get("name"),
        "image": row.get("data", {}).get("mainImage", {}).get("original_url")
    })

with open("sneaker_clusters.json", "w") as f:
    json.dump(groups, f, indent=2)

print(" Clustering complete! Saved to sneaker_clusters.json.")


 Clustering complete! Saved to sneaker_clusters.json.


We may open the json file to inspect the clusters

In [71]:
# Load saved clusters
with open("sneaker_clusters.json", "r") as f:
    clusters = json.load(f)

# Print a summary
print(f"Total groups (including noise): {len(clusters)}\n")

for cluster_id, items in clusters.items():
    print(f"\n Cluster {cluster_id} — {len(items)} items")
    for item in items[:3]:  # preview first 3
        print(f"  - {item['name']} ({item['product_id']})")


Total groups (including noise): 4


 Cluster 1 — 5486 items
  - Gel-rocket 11 mens volleyball/indoor shoes (d96975c5-0bea-41f6-a4a8-fdac53f374f3)
  - Mens filmore suede (6f302b78-9a86-484e-894f-bb39220931a3)
  - React vision worldwide mens (6b010b1d-7475-44dc-ad56-502769d2e742)

 Cluster -1 — 187 items
  - Adidas response (70006973-aa9f-4176-9f38-0ba2c5a1b06f)
  - Under armour ua m phantom 1 reissue (55dc4733-2492-4798-bf4e-63c6e39817b4)
  - Vans men atwood canvas (fdd0eed9-9e05-4ca9-a832-b9cca492151d)

 Cluster 0 — 3 items
  - Cortez womens shoes (5ea92e24-5ea6-4d33-bc51-73ef7e73be6c)
  - Beverly hills polo club women’s lifestyle wedge trainer sneakers with gold detailing (b9720763-1a33-44c6-a8d2-b8151bfee3df)
  - All star cruise (3b8c07ee-252a-4227-b990-3b4131500735)

 Cluster 2 — 3 items
  - New balance (229edeab-2f66-466b-9d6d-78165a285bdf)
  - B202111 (006be3d1-32d8-4f55-9d77-6043261a6a75)
  - Reverse sneakers (22429e41-14d9-419f-beee-04c497566eaf)


### We need a more strict method -- we can proceed with the following logic -- assume that variants/ identical products definitely have the same brand. 
### Therefore we may group by brand and then split those groups into clusters using embeddings -- this method can be more strict / loose depending on how many group-keys we use.

### Hence the attempt below.

## An attempt to find variants using both groupby operations and textual embeddings as well as image embeddings (only brand and category) (call this method *)

In [74]:

# === Step 1: Load metadata and image embeddings ===
df_data = pd.read_json("sneakers_sample_19062025.txt", lines=False)

# Flatten fields
df_data["brand"] = df_data["data"].apply(lambda x: x.get("brand") if isinstance(x, dict) else "")
df_data["color"] = df_data["data"].apply(lambda x: x.get("color") if isinstance(x, dict) else "")
df_data["category"] = df_data["data"].apply(lambda x: x.get("category") if isinstance(x, dict) else "")
df_data["name [en_GB]"] = df_data["data"].apply(lambda x: x.get("name [en_GB]") if isinstance(x, dict) else "")
df_data["shortDescription [en_GB]"] = df_data["data"].apply(lambda x: x.get("shortDescription") if isinstance(x, dict) else "")

# Load image vectors
image_vecs_all = np.load("image_features.npy")
df_data = df_data.reset_index(drop=True).iloc[:len(image_vecs_all)]

# === Step 2: Grouping keys (ONLY brand and category for Method 1★) ===
group_keys = ['brand', 'category']
group_sizes = df_data.groupby(group_keys).size().reset_index(name='count')
duplicates = group_sizes[group_sizes["count"] > 1]
identical_groups = df_data.merge(duplicates[group_keys], on=group_keys, how="inner")
identical_groups["product_group_id"] = identical_groups.groupby(group_keys).ngroup()

# === Step 3: Encode text with sentence transformer ===
text_model = SentenceTransformer("all-MiniLM-L6-v2")    ## contextual embeddings! better

# === Step 4: Cluster each metadata-identical group ===
results_all = {}
method1stars_counter = count(start=0)
method1stars_map = {}
df_data["method1stars"] = pd.NA

for group_id, group_df in identical_groups.groupby("product_group_id"):
    indices = group_df.index.tolist()

    # Prepare text input
    names = group_df["data"].apply(lambda x: x.get("name [en_GB]", "") if isinstance(x, dict) else "")
    brands = group_df["data"].apply(lambda x: x.get("brand", "") if isinstance(x, dict) else "")
    colors = group_df["data"].apply(lambda x: x.get("color", "") if isinstance(x, dict) else "")
    descriptions = group_df["data"].apply(lambda x: x.get("shortDescription", "") if isinstance(x, dict) else "")
    texts = (names + " " + brands + " " + colors + " " + descriptions).fillna("").tolist()
    text_vecs = text_model.encode(texts, normalize_embeddings=True)

    # Prepare image vectors
    img_vecs = normalize(image_vecs_all[indices])

    # Combine image and text features
    alpha = 0.6   # as before -- this could be tuned later
    combined_vecs = normalize(np.hstack([alpha * img_vecs, (1 - alpha) * text_vecs]))

    # Run HDBSCAN clustering
    clusterer = hdbscan.HDBSCAN(min_cluster_size=2, metric='euclidean')
    labels = clusterer.fit_predict(combined_vecs)

    # Assign unique label to each noise point
    unique_id_gen = itertools.count(start=labels.max() + 1)
    labels = [label if label != -1 else next(unique_id_gen) for label in labels]

    # Assign consistent cluster ID (method1stars)
    for idx, label in zip(indices, labels):
        key = (group_id, label)
        if key not in method1stars_map:
            method1stars_map[key] = next(method1stars_counter)
        df_data.loc[idx, "method1stars"] = method1stars_map[key]

    # Optional: Save for visualization
    grouped_json = defaultdict(list)
    for row, label in zip(group_df.itertuples(), labels):
        grouped_json[int(label)].append({
            "product_id": row.mirakl_product_id,
            "name": row.data.get("name [en_GB]", "") if isinstance(row.data, dict) else "",
            "image": row.data.get("mainImage", {}).get("original_url") if isinstance(row.data, dict) else ""
        })
    results_all[f"group_{group_id}"] = grouped_json

# === Step 5: Save JSON and DataFrame === #### so as to not recalculate
with open("clustered_duplicate_groups_1stars.json", "w") as f:
    json.dump(results_all, f, indent=2)

df_data.to_csv("df_with_method1stars.csv", index=False)
print("Done! Method 1★ clusters assigned and exported.")


Done! Method 1★ clusters assigned and exported.


## Evaluating the method above (method *)

In [76]:
# === Load data ===
df_clustered = pd.read_csv("df_with_method1stars.csv")
df_gt = pd.read_csv("sample.csv")

# === Merge with ground truth ===
df = df_clustered.merge(df_gt[["mirakl_product_id", "manual_group_id"]], on="mirakl_product_id", how="inner")

# === Drop NaNs and ensure correct types ===
df = df.dropna(subset=["method1stars"])
df["method1stars"] = df["method1stars"].astype(int)
df["manual_group_id"] = df["manual_group_id"].astype(int)

# === 1. Cluster Purity ===
def compute_cluster_purity(df, cluster_col, gt_col):
    cluster_purities = []
    total = 0
    for _, group in df.groupby(cluster_col):
        most_common = group[gt_col].value_counts().iloc[0]
        cluster_purities.append(most_common)
        total += len(group)
    return np.sum(cluster_purities) / total

purity = compute_cluster_purity(df, cluster_col="method1stars", gt_col="manual_group_id")

# === 2. Map predicted labels from clustering ===
majority_map = (
    df.groupby("method1stars")["manual_group_id"]
    .agg(lambda x: x.value_counts().idxmax())
    .to_dict()
)
df["predicted_manual_group_id"] = df["method1stars"].map(majority_map)
df = df.dropna(subset=["predicted_manual_group_id"])
df["predicted_manual_group_id"] = df["predicted_manual_group_id"].astype(int)

# === 3. Pairwise Confusion Matrix ===
conf_matrix = pair_confusion_matrix(df["manual_group_id"], df["predicted_manual_group_id"])
TP = conf_matrix[1, 1]
FP = conf_matrix[0, 1]
FN = conf_matrix[1, 0]
TN = conf_matrix[0, 0]

# === Compute metrics ===
micro_precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
micro_accuracy = (TP + TN) / (TP + FP + FN + TN)

# === Print all results ===
print(f" Method 1★ Cluster Purity: {purity:.4f}")
print(f" Method 1★ Micro-average Precision (Pairwise): {micro_precision:.4f}")
print(f" Method 1★ Micro-average Accuracy (Pairwise): {micro_accuracy:.4f}")


 Method 1★ Cluster Purity: 0.4759
 Method 1★ Micro-average Precision (Pairwise): 0.0154
 Method 1★ Micro-average Accuracy (Pairwise): 0.9708


### Precision is much lower than we'd like because we prefer stricter methods (we care much more about minimising false negatives - hence why cluster purity is an important metric in our goal)

## Visualising the results of method *

In [79]:
# === Load & Visualize Clusters by Group and Cluster ID ===
# Load clustered results (each metadata group clustered separately)
with open("clustered_duplicate_groups.json", "r") as f:
    clusters_by_group = json.load(f)

# Visualize clusters in notebook
def show_group_cluster(group_id, cluster_id=None, max_items=6):
    group_key = f"group_{group_id}"
    if group_key not in clusters_by_group:
        print(f" Group {group_id} not found.")
        return

    clusters = clusters_by_group[group_key]
    # FIX: don't convert keys to string
    cluster_ids = [cluster_id] if cluster_id is not None else list(clusters.keys())

    for cid in cluster_ids:
        if str(cid) not in clusters:
            print(f"  Cluster {cid} not found in {group_key}")
            continue
        items = clusters[str(cid)][:max_items]
        html = f"<h3 style='margin-top:20px'>Group {group_id} – Cluster {cid} ({len(clusters[str(cid)])} items)</h3><div style='display:flex;flex-wrap:wrap'>"
        for item in items:
            name = item.get('name', '')
            img = item.get('image', '')
            html += f"""
                <div style='margin:10px;text-align:center'>
                    <img src='{img}' width='120' style='border-radius:8px'><br>
                    <div style='max-width:120px;font-size:12px'>{name}</div>
                </div>
            """
        html += "</div>"
        display(HTML(html))


In [80]:
# 1. Show all clusters for first 3 groups
for gid in range(3):
    show_group_cluster(gid)

# 2. Show noise cluster only
for gid in range(3):
    show_group_cluster(gid, cluster_id=-1)

# 3. Show cluster 0 where it exists
for gid in range(len(clusters_by_group)):
    gkey = f"group_{gid}"
    if gkey in clusters_by_group and "0" in clusters_by_group[gkey]:
        show_group_cluster(gid, cluster_id=0)

# 4. Show largest cluster in each group
for gid in range(3):
    gkey = f"group_{gid}"
    if gkey in clusters_by_group:
        largest = max(clusters_by_group[gkey].items(), key=lambda x: len(x[1]))[0]
        show_group_cluster(gid, cluster_id=int(largest))

# 5. Show all clusters that have exactly 2 items
for gid in range(len(clusters_by_group)):
    gkey = f"group_{gid}"
    if gkey in clusters_by_group:
        for cid, items in clusters_by_group[gkey].items():
            if len(items) == 2:
                show_group_cluster(gid, cluster_id=int(cid))

## This method is still not strict enough - clusters non similar items together. We can be more strict with the group keys.


## An attempt to find variants using both groupby operations and textual embeddings as well as image embeddings (only brand and category and name en_gb)(call this method **) 

In [83]:
# === Step 1: Load metadata and image embeddings ===
df_data = pd.read_json("sneakers_sample_19062025.txt", lines=False)

# Flatten fields
df_data["brand"] = df_data["data"].apply(lambda x: x.get("brand") if isinstance(x, dict) else "")
df_data["color"] = df_data["data"].apply(lambda x: x.get("color") if isinstance(x, dict) else "")
df_data["category"] = df_data["data"].apply(lambda x: x.get("category") if isinstance(x, dict) else "")
df_data["name [en_GB]"] = df_data["data"].apply(lambda x: x.get("name [en_GB]") if isinstance(x, dict) else "")
df_data["shortDescription [en_GB]"] = df_data["data"].apply(lambda x: x.get("shortDescription") if isinstance(x, dict) else "")

# Load image vectors
image_vecs_all = np.load("image_features.npy")
df_data = df_data.reset_index(drop=True).iloc[:len(image_vecs_all)]

# === Step 2: Grouping keys for detecting metadata duplicates ===
group_keys = ['brand', 'category', 'name [en_GB]']
group_sizes = df_data.groupby(group_keys).size().reset_index(name='count')
duplicates = group_sizes[group_sizes["count"] > 1]
identical_groups = df_data.merge(duplicates[group_keys], on=group_keys, how="inner")
identical_groups["product_group_id"] = identical_groups.groupby(group_keys).ngroup()

# === Step 3: Encode text with sentence transformer ===
text_model = SentenceTransformer("all-MiniLM-L6-v2")

# === Step 4: Cluster each metadata-identical group ===
results_all = {}
method2stars_counter = count(start=0)
method2stars_map = {}
df_data["method2stars"] = pd.NA

for group_id, group_df in identical_groups.groupby("product_group_id"):
    indices = group_df.index.tolist()

    # Prepare text input
    names = group_df["data"].apply(lambda x: x.get("name [en_GB]", "") if isinstance(x, dict) else "")
    brands = group_df["data"].apply(lambda x: x.get("brand", "") if isinstance(x, dict) else "")
    colors = group_df["data"].apply(lambda x: x.get("color", "") if isinstance(x, dict) else "")
    descriptions = group_df["data"].apply(lambda x: x.get("shortDescription", "") if isinstance(x, dict) else "")
    texts = (names + " " + brands + " " + colors + " " + descriptions).fillna("").tolist()
    text_vecs = text_model.encode(texts, normalize_embeddings=True)

    # Prepare image vectors
    img_vecs = normalize(image_vecs_all[indices])

    # Combine image and text features
    alpha = 0.6
    combined_vecs = normalize(np.hstack([alpha * img_vecs, (1 - alpha) * text_vecs]))

    # Run HDBSCAN clustering
    clusterer = hdbscan.HDBSCAN(min_cluster_size=2, metric='euclidean')
    labels = clusterer.fit_predict(combined_vecs)

    # Assign unique label to each noise point
    unique_id_gen = itertools.count(start=labels.max() + 1)
    labels = [label if label != -1 else next(unique_id_gen) for label in labels]

    # Assign consistent cluster ID (method2stars)   - this will be useful for evaluation later
    for idx, label in zip(indices, labels):
        key = (group_id, label)
        if key not in method2stars_map:
            method2stars_map[key] = next(method2stars_counter)
        df_data.loc[idx, "method2stars"] = method2stars_map[key]

    # Save for visualization 
    grouped_json = defaultdict(list)
    for row, label in zip(group_df.itertuples(), labels):
        grouped_json[int(label)].append({
            "product_id": row.mirakl_product_id,
            "name": row.data.get("name [en_GB]", "") if isinstance(row.data, dict) else "",
            "image": row.data.get("mainImage", {}).get("original_url") if isinstance(row.data, dict) else ""
        })
    results_all[f"group_{group_id}"] = grouped_json

# === Step 5: Save JSON and DataFrame ===
with open("clustered_duplicate_groups_2stars.json", "w") as f:
    json.dump(results_all, f, indent=2)

df_data.to_csv("df_with_method2stars.csv", index=False)
print("Done! Clusters assigned and exported.")


Done! Clusters assigned and exported.


## Evaluating method **

In [220]:

# === Load data ===
df = pd.read_csv("df_with_method2stars.csv")
df_gt = pd.read_csv("sample.csv")

# === Merge with ground truth ===
df = df.merge(df_gt[["mirakl_product_id", "manual_group_id"]], on="mirakl_product_id", how="inner")

# === Drop NaNs and ensure correct types ===
df = df.dropna(subset=["method2stars", "manual_group_id"])
df["method2stars"] = df["method2stars"].astype(int)
df["manual_group_id"] = df["manual_group_id"].astype(int)

# === 1. Cluster Purity ===

purity = compute_cluster_purity(df, cluster_col="method2stars", gt_col="manual_group_id")

# === 2. Map predicted labels from clustering ===
majority_map = (
    df.groupby("method2stars")["manual_group_id"]
    .agg(lambda x: x.value_counts().idxmax())
    .to_dict()
)
df["predicted_manual_group_id"] = df["method2stars"].map(majority_map)
df = df.dropna(subset=["predicted_manual_group_id"])
df["predicted_manual_group_id"] = df["predicted_manual_group_id"].astype(int)

# === 3. Pairwise Confusion Matrix ===
conf_matrix = pair_confusion_matrix(df["manual_group_id"], df["predicted_manual_group_id"])
TP = conf_matrix[1, 1]
FP = conf_matrix[0, 1]
FN = conf_matrix[1, 0]
TN = conf_matrix[0, 0]

# === Compute metrics ===
micro_precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
micro_accuracy = (TP + TN) / (TP + FP + FN + TN)

# === Print all results ===
print(f" Method 2★ Cluster Purity: {purity:.4f}")
print(f" Method 2★ Micro-average Precision (Pairwise): {micro_precision:.4f}")
print(f" Method 2★ Micro-average Accuracy (Pairwise): {micro_accuracy:.9f}")   # in 4 decimal places this gave 1.0000 - concerning - however not exactly 1


 Method 2★ Cluster Purity: 0.9965
 Method 2★ Micro-average Precision (Pairwise): 0.9700
 Method 2★ Micro-average Accuracy (Pairwise): 0.999950840


## Visualising method **

In [87]:
# === Load & Visualize Clusters by Group and Cluster ID ===

# Load clustered results (each metadata group clustered separately)
with open("clustered_duplicate_groups_2.json", "r") as f:
    clusters_by_group = json.load(f)



In [88]:
# 1. Show all clusters for first 3 groups
for gid in range(3):
    show_group_cluster(gid)

# 2. Show noise cluster only
for gid in range(3):
    show_group_cluster(gid, cluster_id=-1)

# 3. Show cluster 0 where it exists
for gid in range(len(clusters_by_group)):
    gkey = f"group_{gid}"
    if gkey in clusters_by_group and "0" in clusters_by_group[gkey]:
        show_group_cluster(gid, cluster_id=0)

# 4. Show largest cluster in each group
for gid in range(3):
    gkey = f"group_{gid}"
    if gkey in clusters_by_group:
        largest = max(clusters_by_group[gkey].items(), key=lambda x: len(x[1]))[0]
        show_group_cluster(gid, cluster_id=int(largest))

# 5. Show all clusters that have exactly 2 items
for gid in range(len(clusters_by_group)):
    gkey = f"group_{gid}"
    if gkey in clusters_by_group:
        for cid, items in clusters_by_group[gkey].items():
            if len(items) >= 2:     #only change
                show_group_cluster(gid, cluster_id=int(cid))

  Cluster -1 not found in group_0
  Cluster -1 not found in group_1
  Cluster -1 not found in group_2


### We can see in the clusters visualised as well that the sneakers clustered together are indeed variants (indicating high cluster purity).

### Tuning alpha or using a different group key combination with more elegant embeddings could lead to an even better clustering. Which eliminates some false negatives, which this method oversees.

## To find identical products (instead of variants) also used color as a group key (method ***) -- Note: This method cannot be evaluated on the sample as the sample contains clusters of variants and not identical sneakers. 

In [91]:
# === Step 1: Load metadata and image embeddings ===
df_data = pd.read_json("sneakers_sample_19062025.txt", lines=False)

# Flatten fields
df_data["brand"] = df_data["data"].apply(lambda x: x.get("brand") if isinstance(x, dict) else "")
df_data["color"] = df_data["data"].apply(lambda x: x.get("color") if isinstance(x, dict) else "")
df_data["category"] = df_data["data"].apply(lambda x: x.get("category") if isinstance(x, dict) else "")
df_data["name [en_GB]"] = df_data["data"].apply(lambda x: x.get("name [en_GB]") if isinstance(x, dict) else "")
df_data["shortDescription [en_GB]"] = df_data["data"].apply(lambda x: x.get("shortDescription") if isinstance(x, dict) else "")

# Load image vectors
image_vecs_all = np.load("image_features.npy")
df_data = df_data.reset_index(drop=True).iloc[:len(image_vecs_all)]

# === Step 2: Grouping keys for detecting metadata duplicates ===
group_keys = ['brand', 'category', 'name [en_GB]','color']
group_sizes = df_data.groupby(group_keys).size().reset_index(name='count')
duplicates = group_sizes[group_sizes["count"] > 1]
identical_groups = df_data.merge(duplicates[group_keys], on=group_keys, how="inner")
identical_groups["product_group_id"] = identical_groups.groupby(group_keys).ngroup()

# === Step 3: Encode text with sentence transformer ===
text_model = SentenceTransformer("all-MiniLM-L6-v2")

# === Step 4: Cluster each metadata-identical group ===
results_all = {}
method3stars_counter = count(start=0)
method3stars_map = {}
df_data["method3stars"] = pd.NA

for group_id, group_df in identical_groups.groupby("product_group_id"):
    indices = group_df.index.tolist()

    # Prepare text input
    names = group_df["data"].apply(lambda x: x.get("name [en_GB]", "") if isinstance(x, dict) else "")
    brands = group_df["data"].apply(lambda x: x.get("brand", "") if isinstance(x, dict) else "")
    colors = group_df["data"].apply(lambda x: x.get("color", "") if isinstance(x, dict) else "")
    descriptions = group_df["data"].apply(lambda x: x.get("shortDescription", "") if isinstance(x, dict) else "")
    texts = (names + " " + brands + " " + colors + " " + descriptions).fillna("").tolist()
    text_vecs = text_model.encode(texts, normalize_embeddings=True)

    # Prepare image vectors
    img_vecs = normalize(image_vecs_all[indices])

    # Combine image and text features
    alpha = 0.6
    combined_vecs = normalize(np.hstack([alpha * img_vecs, (1 - alpha) * text_vecs]))

    # Run HDBSCAN clustering
    clusterer = hdbscan.HDBSCAN(min_cluster_size=2, metric='euclidean')
    labels = clusterer.fit_predict(combined_vecs)

    # Assign unique label to each noise point
    unique_id_gen = itertools.count(start=labels.max() + 1)
    labels = [label if label != -1 else next(unique_id_gen) for label in labels]

    # Assign consistent cluster ID (method3stars)
    for idx, label in zip(indices, labels):
        key = (group_id, label)
        if key not in method3stars_map:
            method3stars_map[key] = next(method3stars_counter)
        df_data.loc[idx, "method3stars"] = method3stars_map[key]

    grouped_json = defaultdict(list)
    for row, label in zip(group_df.itertuples(), labels):
        grouped_json[int(label)].append({
            "product_id": row.mirakl_product_id,
            "name": row.data.get("name [en_GB]", "") if isinstance(row.data, dict) else "",
            "image": row.data.get("mainImage", {}).get("original_url") if isinstance(row.data, dict) else ""
        })
    results_all[f"group_{group_id}"] = grouped_json

# === Step 5: Save JSON and DataFrame ===
with open("clustered_duplicate_groups_3stars.json", "w") as f:
    json.dump(results_all, f, indent=2)

df_data.to_csv("df_with_method3stars.csv", index=False)
print("Done! Method 3★ clustering saved.")


Done! Method 3★ clustering saved.


## Visualising this method

In [93]:
# === Load & Visualize Clusters by Group and Cluster ID ===

# Load clustered results (each metadata group clustered separately)
with open("identical_groups.json", "r") as f:
    clusters_by_group = json.load(f)



In [94]:
# 1. Show all clusters for first 3 groups
for gid in range(3):
    show_group_cluster(gid)

# 2. Show noise cluster only
for gid in range(3):
    show_group_cluster(gid, cluster_id=-1)

# 3. Show cluster 0 where it exists
for gid in range(len(clusters_by_group)):
    gkey = f"group_{gid}"
    if gkey in clusters_by_group and "0" in clusters_by_group[gkey]:
        show_group_cluster(gid, cluster_id=0)

# 4. Show largest cluster in each group
for gid in range(3):
    gkey = f"group_{gid}"
    if gkey in clusters_by_group:
        largest = max(clusters_by_group[gkey].items(), key=lambda x: len(x[1]))[0]
        show_group_cluster(gid, cluster_id=int(largest))

# 5. Show all clusters that have exactly 2 items
for gid in range(len(clusters_by_group)):
    gkey = f"group_{gid}"
    if gkey in clusters_by_group:
        for cid, items in clusters_by_group[gkey].items():
            if len(items) == 2:
                show_group_cluster(gid, cluster_id=int(cid))

## From manual inspection this has some promising results. However, we have no way of evaluating this method as it aims to cluster identical sneakers.

## Method **** : imitate method ** but on grayscaled images

In [97]:
# ### the below code creates and stores grayscaled embeddings so we do not have to generate them every time - just once and then load
# ### can comment this out later 

# # === Setup model ===
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
# resnet = torch.nn.Sequential(*list(resnet.children())[:-1]).to(device).eval()

# gray_transform = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.Grayscale(num_output_channels=3),  # Still 3 channels
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                          std=[0.229, 0.224, 0.225])
# ])

# # === Extract grayscale image embeddings ===
# embeddings = []
# valid_indices = []

# for idx, row in df_data.iterrows():
#     try:
#         url = row["data"].get("mainImage", {}).get("original_url")
#         if not isinstance(url, str) or not url.startswith("http"):
#             print(f" Skipping idx {idx} – invalid URL")
#             continue
#         response = requests.get(url, timeout=10)
#         if response.status_code != 200 or b"<html" in response.content[:100].lower():
#             print(f" Bad response for idx {idx}")
#             continue
#         img = Image.open(BytesIO(response.content)).convert("RGB")
#         gray_tensor = gray_transform(img).unsqueeze(0).to(device)
#         with torch.no_grad():
#             vec = resnet(gray_tensor).squeeze().cpu().numpy()
#         embeddings.append(vec)
#         valid_indices.append(idx)
#         print(f"Done idx {idx}")
#     except Exception as e:
#         print(f" Exception at idx {idx}: {e}")
#         continue

# if not embeddings:
#     raise ValueError("No valid grayscale image embeddings were extracted. Check image URLs or connection.")

# # === Save outputs ===
# embeddings = np.vstack(embeddings)
# np.save("grayscale_clip_embeddings.npy", embeddings)
# pd.Series(valid_indices).to_csv("grayscale_clip_valid_indices.csv", index=False)

# print(" Saved grayscale embeddings and indices.")


In [98]:
# === Step 1: Load metadata and grayscale embeddings ===
df_data = pd.read_json("sneakers_sample_19062025.txt", lines=False)

# Flatten fields
df_data["brand"] = df_data["data"].apply(lambda x: x.get("brand") if isinstance(x, dict) else "")
df_data["color"] = df_data["data"].apply(lambda x: x.get("color") if isinstance(x, dict) else "")
df_data["category"] = df_data["data"].apply(lambda x: x.get("category") if isinstance(x, dict) else "")
df_data["name [en_GB]"] = df_data["data"].apply(lambda x: x.get("name [en_GB]") if isinstance(x, dict) else "")
df_data["shortDescription [en_GB]"] = df_data["data"].apply(lambda x: x.get("shortDescription") if isinstance(x, dict) else "")

# Load grayscale embeddings and valid indices
grayscale_embeddings = np.load("grayscale_clip_embeddings.npy")
valid_indices = pd.read_csv("grayscale_clip_valid_indices.csv", header=None)[0].tolist()

# Align df_data
df_valid = df_data.loc[valid_indices].copy().reset_index(drop=True)

# === Step 2: Grouping keys for detecting metadata duplicates ===
group_keys = ['brand', 'category', 'name [en_GB]']
group_sizes = df_valid.groupby(group_keys).size().reset_index(name='count')
duplicates = group_sizes[group_sizes["count"] > 1]
identical_groups = df_valid.merge(duplicates[group_keys], on=group_keys, how="inner")
identical_groups["product_group_id"] = identical_groups.groupby(group_keys).ngroup()

# === Step 3: Encode text with sentence transformer ===
text_model = SentenceTransformer("all-MiniLM-L6-v2")

# === Step 4: Cluster each metadata-identical group ===
results_all = {}
method2stars_counter = count(start=0)
method2stars_map = {}
df_valid["method2stars"] = pd.NA

for group_id, group_df in identical_groups.groupby("product_group_id"):
    indices = group_df.index.tolist()
    
    # Text features
    names = group_df["data"].apply(lambda x: x.get("name [en_GB]", "") if isinstance(x, dict) else "")
    brands = group_df["data"].apply(lambda x: x.get("brand", "") if isinstance(x, dict) else "")
    colors = group_df["data"].apply(lambda x: x.get("color", "") if isinstance(x, dict) else "")
    descriptions = group_df["data"].apply(lambda x: x.get("shortDescription", "") if isinstance(x, dict) else "")
    texts = (names + " " + brands + " " + colors + " " + descriptions).fillna("").tolist()
    text_vecs = text_model.encode(texts, normalize_embeddings=True)

    # Image vectors for this group (from grayscale embeddings)
    img_vecs = normalize(grayscale_embeddings[indices])

    # Combine image and text vectors
    alpha = 0.6  # weight for image vectors
    combined_vecs = normalize(np.hstack([alpha * img_vecs, (1 - alpha) * text_vecs]))

    # Run HDBSCAN clustering
    clusterer = hdbscan.HDBSCAN(min_cluster_size=2, metric='euclidean')
    labels = clusterer.fit_predict(combined_vecs)

    # Handle noise points
    unique_id_gen = itertools.count(start=labels.max() + 1)
    labels = [label if label != -1 else next(unique_id_gen) for label in labels]

    # Assign consistent cluster ID (method2stars)
    for idx, label in zip(indices, labels):
        key = (group_id, label)
        if key not in method2stars_map:
            method2stars_map[key] = next(method2stars_counter)
        df_valid.loc[idx, "method2stars"] = method2stars_map[key]

    # Save grouped results for visualization
    grouped_json = defaultdict(list)
    for row, label in zip(group_df.itertuples(), labels):
        grouped_json[int(label)].append({
            "product_id": row.mirakl_product_id,
            "name": row.data.get("name [en_GB]", "") if isinstance(row.data, dict) else "",
            "image": row.data.get("mainImage", {}).get("original_url") if isinstance(row.data, dict) else ""
        })
    results_all[f"group_{group_id}"] = grouped_json

# === Step 5: Save JSON and DataFrame ===
with open("clustered_duplicate_groups_2stars_gray.json", "w") as f:
    json.dump(results_all, f, indent=2)

df_valid.to_csv("df_with_method2stars_gray.csv", index=False)
print("Done! Grayscale clusters assigned and exported.")


Done! Grayscale clusters assigned and exported.


## Visualising method ****

In [100]:

# === Load saved cluster results ===
with open("clustered_duplicate_groups_2stars_gray.json", "r") as f:
    clusters_by_group = json.load(f)

# === Visualize all clusters ===
max_items = 6  # Number of items to display per cluster

for group_key, clusters in clusters_by_group.items():
    for cid_str, items in clusters.items():
        html = f"<h3 style='margin-top:20px'>{group_key} – Cluster {cid_str} ({len(items)} items)</h3><div style='display:flex;flex-wrap:wrap'>"
        for item in items[:max_items]:
            name = item.get('name', '')
            img = item.get('image', '')
            html += f"""
                <div style='margin:10px;text-align:center'>
                    <img src='{img}' width='120' style='border-radius:8px'><br>
                    <div style='max-width:120px;font-size:12px'>{name}</div>
                </div>
            """
        html += "</div>"
        display(HTML(html))


IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [101]:
### keep this function! necessary for future visualisations 
def visualize_clusters(json_path="clustered_duplicate_groups_2stars_gray.json", min_items=2, max_items_display=6):
    """
    Displays image clusters from a JSON file, showing only clusters with more than `min_items` items.
    
    Parameters:
        json_path (str): Path to the JSON file containing grouped clusters.
        min_items (int): Minimum number of items required in a cluster to be shown.
        max_items_display (int): Maximum number of images to display per cluster.
    """
    with open(json_path, "r") as f:
        clusters_by_group = json.load(f)

    for group_key, clusters in clusters_by_group.items():
        for cid_str, items in clusters.items():
            if len(items) < min_items:
                continue  # Skip small clusters
            html = f"<h3 style='margin-top:20px'>{group_key} – Cluster {cid_str} ({len(items)} items)</h3><div style='display:flex;flex-wrap:wrap'>"
            for item in items[:max_items_display]:
                name = item.get('name', '')
                img = item.get('image', '')
                html += f"""
                    <div style='margin:10px;text-align:center'>
                        <img src='{img}' width='120' style='border-radius:8px'><br>
                        <div style='max-width:120px;font-size:12px'>{name}</div>
                    </div>
                """
            html += "</div>"
            display(HTML(html))


In [102]:
visualize_clusters("clustered_duplicate_groups_2stars_gray.json", min_items=3, max_items_display=6)


## Evaluating method ****

In [104]:
# === Load grayscale-based clustering result and ground truth ===
df = pd.read_csv("df_with_method2stars_gray.csv")
df_gt = pd.read_csv("sample.csv")

# === Merge with ground truth ===
df = df.merge(df_gt[["mirakl_product_id", "manual_group_id"]], on="mirakl_product_id", how="inner")

# === Drop NaNs and ensure correct types ===
df = df.dropna(subset=["method2stars", "manual_group_id"])
df["method2stars"] = df["method2stars"].astype(int)
df["manual_group_id"] = df["manual_group_id"].astype(int)

# === 1. Cluster Purity ===

purity = compute_cluster_purity(df, "method2stars", "manual_group_id")

# === 2. Map predicted labels from clustering ===
majority_map = (
    df.groupby("method2stars")["manual_group_id"]
    .agg(lambda x: x.value_counts().idxmax())
    .to_dict()
)
df["predicted_manual_group_id"] = df["method2stars"].map(majority_map)
df = df.dropna(subset=["predicted_manual_group_id"])
df["predicted_manual_group_id"] = df["predicted_manual_group_id"].astype(int)

# === 3. Pairwise Confusion Matrix ===
conf_matrix = pair_confusion_matrix(df["manual_group_id"], df["predicted_manual_group_id"])
TP = conf_matrix[1, 1]
FP = conf_matrix[0, 1]
FN = conf_matrix[1, 0]
TN = conf_matrix[0, 0]

# === Compute metrics ===
micro_precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
micro_accuracy = (TP + TN) / (TP + FP + FN + TN)

# === Print all results ===
print(f" Method 4★ Cluster Purity (grayscale): {purity:.4f}")
print(f" Method 4★ Micro-average Precision (Pairwise): {micro_precision:.4f}")
print(f" Method 4★ Micro-average Accuracy (Pairwise): {micro_accuracy:.4f}")


 Method 4★ Cluster Purity (grayscale): 0.9843
 Method 4★ Micro-average Precision (Pairwise): 0.9015
 Method 4★ Micro-average Accuracy (Pairwise): 0.9998


### This method seems to perform very well, but does not outperform method **

<!-- ## Method ***** -->

In [107]:
# # === Load metadata and embeddings ===
# df_data = pd.read_json("sneakers_sample_19062025.txt", lines=False)
# grayscale_embeddings = np.load("grayscale_clip_embeddings.npy")
# valid_indices = pd.read_csv("grayscale_clip_valid_indices.csv", header=None)[0].tolist()

# # === Fix index mismatch if needed ===
# if len(valid_indices) > grayscale_embeddings.shape[0]:
#     valid_indices = valid_indices[:grayscale_embeddings.shape[0]]
# elif len(valid_indices) < grayscale_embeddings.shape[0]:
#     grayscale_embeddings = grayscale_embeddings[:len(valid_indices)]

# # === Align df_data with embeddings ===
# df_valid = df_data.iloc[valid_indices].copy().reset_index(drop=True)
# df_valid["grayscale_row"] = np.arange(len(df_valid))

# # === Flatten text fields ===
# df_valid["brand"] = df_valid["data"].apply(lambda x: x.get("brand") if isinstance(x, dict) else "")
# df_valid["color"] = df_valid["data"].apply(lambda x: x.get("color") if isinstance(x, dict) else "")
# df_valid["category"] = df_valid["data"].apply(lambda x: x.get("category") if isinstance(x, dict) else "")
# df_valid["name"] = df_valid["data"].apply(lambda x: x.get("name [en_GB]") if isinstance(x, dict) else "")
# df_valid["desc"] = df_valid["data"].apply(lambda x: x.get("shortDescription") if isinstance(x, dict) else "")

# # === Build text input ===
# texts = (df_valid["name"].fillna("") + " " +
#          df_valid["brand"].fillna("") + " " +
#          df_valid["color"].fillna("") + " " +
#          df_valid["desc"].fillna("")).tolist()

# # === Encode text ===
# text_model = SentenceTransformer("all-MiniLM-L6-v2")
# text_vecs = text_model.encode(texts, normalize_embeddings=True)

# # === Normalize image vectors ===
# img_vecs = normalize(grayscale_embeddings)

# # === Combine vectors ===
# alpha = 0.6
# combined_vecs = normalize(np.hstack([alpha * img_vecs, (1 - alpha) * text_vecs]))

# # === Cluster all products together ===
# clusterer = hdbscan.HDBSCAN(min_cluster_size=2, metric='euclidean')
# labels = clusterer.fit_predict(combined_vecs)

# # === Replace noise points with unique cluster IDs ===
# unique_id_gen = itertools.count(start=labels.max() + 1)
# labels = [label if label != -1 else next(unique_id_gen) for label in labels]
# df_valid["method2stars"] = labels

# # Save clustered file
# df_valid.to_csv("df_with_method2stars_global.csv", index=False)
# print(" Clustered full dataset and saved to df_with_method2stars_global.csv")

<!-- ## Evaluating Method ***** -->

## Comparing this to using no images - just text metadata embeddings (identical products, include colour as a group key)

In [110]:
# === Step 1: Load metadata ===
df_data = pd.read_json("sneakers_sample_19062025.txt", lines=False)

# Flatten fields for grouping and processing
df_data["brand"] = df_data["data"].apply(lambda x: x.get("brand") if isinstance(x, dict) else "")
df_data["color"] = df_data["data"].apply(lambda x: x.get("color") if isinstance(x, dict) else "")
df_data["category"] = df_data["data"].apply(lambda x: x.get("category") if isinstance(x, dict) else "")
df_data["name [en_GB]"] = df_data["data"].apply(lambda x: x.get("name [en_GB]") if isinstance(x, dict) else "")
df_data["shortDescription [en_GB]"] = df_data["data"].apply(lambda x: x.get("shortDescription") if isinstance(x, dict) else "")

# === Step 2: Group by metadata keys (excluding image info) ===
group_keys = ['color', 'brand', 'category', 'name [en_GB]']
group_sizes = df_data.groupby(group_keys).size().reset_index(name='count')
duplicates = group_sizes[group_sizes["count"] > 1]
identical_groups = df_data.merge(duplicates[group_keys], on=group_keys, how="inner")
identical_groups["product_group_id"] = identical_groups.groupby(group_keys).ngroup()

# === Step 3: Prepare sentence transformer ===
text_model = SentenceTransformer("all-MiniLM-L6-v2")

# === Step 4: Cluster text-only features per group ===
results_all = {}

for group_id, group_df in identical_groups.groupby("product_group_id"):
    # Extract text
    names = group_df["data"].apply(lambda x: x.get("name [en_GB]", "") if isinstance(x, dict) else "")
    brands = group_df["data"].apply(lambda x: x.get("brand", "") if isinstance(x, dict) else "")
    colors = group_df["data"].apply(lambda x: x.get("color", "") if isinstance(x, dict) else "")
    descriptions = group_df["data"].apply(lambda x: x.get("shortDescription", "") if isinstance(x, dict) else "")
    texts = (names + " " + brands + " " + colors + " " + descriptions).fillna("").tolist()
    text_vecs = text_model.encode(texts, normalize_embeddings=True)

    # Cluster text embeddings
    clusterer = hdbscan.HDBSCAN(min_cluster_size=2, metric='euclidean')
    labels = clusterer.fit_predict(text_vecs)
    group_df = group_df.copy()
    group_df["cluster_id"] = labels

    # Save clusters as JSON
    grouped_json = defaultdict(list)
    for _, row in group_df.iterrows():
        grouped_json[int(row["cluster_id"])].append({
            "product_id": row.get("mirakl_product_id"),
            "name": row.get("data", {}).get("name [en_GB]", ""),
            "image": row.get("data", {}).get("mainImage", {}).get("original_url")
        })

    results_all[f"group_{group_id}"] = grouped_json

# === Step 5: Save results ===
with open("text_only_clustered_groups.json", "w") as f:
    json.dump(results_all, f, indent=2)

print("Text-only clustering complete!")
# === Step 5b: Create flat DataFrame with predicted labels ===
flat_records = []
for group_key, clusters in results_all.items():
    for cluster_id, items in clusters.items():
        for item in items:
            flat_records.append({
                "mirakl_product_id": item["product_id"],
                "method_metadata_only1": f"{group_key}_cluster{cluster_id}"
            })

df_predicted_labels = pd.DataFrame(flat_records)

# Save this too for later use
df_predicted_labels.to_csv("df_with_method_metadata_only1.csv", index=False)



Text-only clustering complete!


## Evaluating this method

In [112]:
# === Step 1: Load clustered result and ground truth ===
df_clustered = pd.read_csv("df_with_method_metadata_only1.csv")
df_gt = pd.read_csv("sample.csv")

# === Step 2: Merge with ground truth ===
df = df_clustered.merge(df_gt[["mirakl_product_id", "manual_group_id"]], on="mirakl_product_id", how="inner")
df = df.dropna(subset=["method_metadata_only1", "manual_group_id"])
df["manual_group_id"] = df["manual_group_id"].astype(int)

# === Step 3: Compute Cluster Purity ===
purity = compute_cluster_purity(df, "method_metadata_only1", "manual_group_id")

# === Step 4: Map clusters to predicted labels ===
majority_map = (
    df.groupby("method_metadata_only1")["manual_group_id"]
    .agg(lambda x: x.value_counts().idxmax())
    .to_dict()
)
df["predicted_manual_group_id"] = df["method_metadata_only1"].map(majority_map)
df = df.dropna(subset=["predicted_manual_group_id"])
df["predicted_manual_group_id"] = df["predicted_manual_group_id"].astype(int)

# === Step 5: Pairwise Confusion Matrix ===
conf_matrix = pair_confusion_matrix(df["manual_group_id"], df["predicted_manual_group_id"])
TP = conf_matrix[1, 1]
FP = conf_matrix[0, 1]
FN = conf_matrix[1, 0]
TN = conf_matrix[0, 0]

# === Step 6: Metrics ===
micro_precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
micro_accuracy = (TP + TN) / (TP + FP + FN + TN)

# === Step 7: Print Results ===
print(f" Method metadata_only1 Cluster Purity: {purity:.4f}")
print(f" Method metadata_only1 Micro-average Precision (Pairwise): {micro_precision:.4f}")
print(f" Method metadata_only1 Micro-average Accuracy (Pairwise): {micro_accuracy:.4f}")


 Method metadata_only1 Cluster Purity: 1.0000
 Method metadata_only1 Micro-average Precision (Pairwise): 1.0000
 Method metadata_only1 Micro-average Accuracy (Pairwise): 1.0000


### These results are suspicious. Does this mean that this method produde only singletons? And therefore is of no use?

In [114]:
print("Unique ground truth groups:", df["manual_group_id"].nunique())
print("Unique predicted clusters:", df["method_metadata_only1"].nunique())


Unique ground truth groups: 254
Unique predicted clusters: 328


In [115]:
# How many clusters contain only one unique ground truth label?
cluster_purity_check = df.groupby("method_metadata_only1")["manual_group_id"].nunique()
print("Clusters with only one unique GT label:", (cluster_purity_check == 1).sum())
print("Total clusters:", len(cluster_purity_check))


Clusters with only one unique GT label: 328
Total clusters: 328


In [116]:
print("Singleton clusters:", (df["method_metadata_only1"].value_counts() == 1).sum())
print("Total clusters:", df["method_metadata_only1"].nunique())


Singleton clusters: 293
Total clusters: 328


## We can see that this doesn't give only singletons, but rather a lot of very small clusters - and many singletons - indicating that although this method gives high cluster purity, it may not be very meaningful.

## Visualising this method

In [119]:
# === Visualize lots of clusters ===

# Show all clusters for first 5 groups
for gid in range(5):
    show_group_cluster(gid)

# Show noise cluster (-1) for first 5 groups
for gid in range(5):
    show_group_cluster(gid, cluster_id=-1)

# Show cluster 0 where it exists
for gid in range(len(clusters_by_group)):
    gkey = f"group_{gid}"
    if gkey in clusters_by_group and "0" in clusters_by_group[gkey]:
        show_group_cluster(gid, cluster_id=0)

# Show largest cluster in each of the first 10 groups
for gid in range(10):
    gkey = f"group_{gid}"
    if gkey in clusters_by_group:
        largest = max(clusters_by_group[gkey].items(), key=lambda x: len(x[1]))[0]
        show_group_cluster(gid, cluster_id=int(largest))

# Show all clusters that have exactly 2 items
for gid in range(len(clusters_by_group)):
    gkey = f"group_{gid}"
    if gkey in clusters_by_group:
        for cid, items in clusters_by_group[gkey].items():
            if len(items) == 2:
                show_group_cluster(gid, cluster_id=int(cid))


  Cluster -1 not found in group_0
  Cluster -1 not found in group_1
  Cluster -1 not found in group_2
  Cluster -1 not found in group_3
  Cluster -1 not found in group_4


In [120]:
# 1. Show all clusters for first 3 groups
for gid in range(3):
    show_group_cluster(gid)

# 2. Show noise cluster only
for gid in range(3):
    show_group_cluster(gid, cluster_id=-1)

# 3. Show cluster 0 where it exists
for gid in range(len(clusters_by_group)):
    gkey = f"group_{gid}"
    if gkey in clusters_by_group and "0" in clusters_by_group[gkey]:
        show_group_cluster(gid, cluster_id=0)

# 4. Show largest cluster in each group
for gid in range(3):
    gkey = f"group_{gid}"
    if gkey in clusters_by_group:
        largest = max(clusters_by_group[gkey].items(), key=lambda x: len(x[1]))[0]
        show_group_cluster(gid, cluster_id=int(largest))

# 5. Show all clusters that have exactly 2 items
for gid in range(len(clusters_by_group)):
    gkey = f"group_{gid}"
    if gkey in clusters_by_group:
        for cid, items in clusters_by_group[gkey].items():
            if len(items) == 2:
                show_group_cluster(gid, cluster_id=int(cid))

  Cluster -1 not found in group_0
  Cluster -1 not found in group_1
  Cluster -1 not found in group_2


## Trying the method above without group-by operations to fix "overfitting"

In [122]:

# === Load data ===
df_data = pd.read_json("sneakers_sample_19062025.txt", lines=False)

# === Extract text features ===
def extract_text(row):
    if isinstance(row, dict):
        name = row.get("name [en_GB]", "")
        brand = row.get("brand", "")
        color = row.get("color", "")
        desc = row.get("shortDescription", "")
        return f"{name} {brand} {color} {desc}".strip()
    return ""

df_data["text"] = df_data["data"].apply(extract_text)

# === Encode with SentenceTransformer ===
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(df_data["text"].tolist(), normalize_embeddings=True)

# === Cluster with HDBSCAN ===
clusterer = hdbscan.HDBSCAN(min_cluster_size=2, metric='euclidean')
labels = clusterer.fit_predict(embeddings)

# === Assign to DataFrame ===
df_data["method_metadata_only2"] = labels  # -1 = noise (ignored in evaluation)

# === Save cluster labels for evaluation ===
df_out = df_data[["mirakl_product_id", "method_metadata_only2"]].copy()
df_out.to_csv("df_with_method_metadata_only2.csv", index=False)

print("method_metadata_only2 clustering complete! Saved to CSV.")


method_metadata_only2 clustering complete! Saved to CSV.


## Evaluating this method

In [124]:

# === Load predictions and ground truth ===
df_clustered = pd.read_csv("df_with_method_metadata_only2.csv")
df_gt = pd.read_csv("sample.csv")

# === Merge and clean ===
df = df_clustered.merge(df_gt[["mirakl_product_id", "manual_group_id"]], on="mirakl_product_id", how="inner")
df = df[df["method_metadata_only2"] != -1]  # Ignore noise points
df["manual_group_id"] = df["manual_group_id"].astype(int)

# === Cluster Purity ===
purity = compute_cluster_purity(df, "method_metadata_only2", "manual_group_id")

# === Map predicted labels ===
majority_map = (
    df.groupby("method_metadata_only2")["manual_group_id"]
    .agg(lambda x: x.value_counts().idxmax())
    .to_dict()
)
df["predicted_manual_group_id"] = df["method_metadata_only2"].map(majority_map)

# === Pairwise Confusion Matrix ===
conf_matrix = pair_confusion_matrix(df["manual_group_id"], df["predicted_manual_group_id"])
TP = conf_matrix[1, 1]
FP = conf_matrix[0, 1]
FN = conf_matrix[1, 0]
TN = conf_matrix[0, 0]

micro_precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
micro_accuracy = (TP + TN) / (TP + FP + FN + TN)

# === Print Results ===
print(f"method_metadata_only2 Cluster Purity: {purity:.4f}")
print(f"method_metadata_only2 Micro-average Precision (Pairwise): {micro_precision:.4f}")
print(f"method_metadata_only2 Micro-average Accuracy (Pairwise): {micro_accuracy:.4f}")


method_metadata_only2 Cluster Purity: 0.8939
method_metadata_only2 Micro-average Precision (Pairwise): 0.7558
method_metadata_only2 Micro-average Accuracy (Pairwise): 0.9992


## Visualising this method

In [126]:
# Load clustered results (optional if in memory already)
df = pd.read_csv("df_with_method_metadata_only2.csv")
df_meta = pd.read_json("sneakers_sample_19062025.txt", lines=False)

# Merge to get image and name for display
df = df.merge(df_meta[["mirakl_product_id", "data"]], on="mirakl_product_id", how="left")

# Convert to visual-friendly format
clusters_flat = {}

for _, row in df.iterrows():
    cid = row["method_metadata_only2"]
    if cid not in clusters_flat:
        clusters_flat[cid] = []
    clusters_flat[cid].append({
        "product_id": row["mirakl_product_id"],
        "name": row["data"].get("name [en_GB]", ""),
        "image": row["data"].get("mainImage", {}).get("original_url", "")
    })


In [127]:

def show_cluster(cluster_id):
    items = clusters_flat.get(cluster_id, [])
    if not items:
        print(f"No items found in cluster {cluster_id}")
        return

    cluster_name = "Noise Cluster" if cluster_id == -1 else f"Cluster {cluster_id}"
    cluster_size = len(items)

    html = f"""
    <style>
        .cluster-header {{
            font-family: Arial, sans-serif;
            font-size: 18px;
            font-weight: bold;
            margin: 20px 0 10px 0;
            color: #222;
        }}
        .cluster-container {{
            display: flex;
            flex-wrap: wrap;
            gap: 20px;
            padding: 10px;
        }}
        .cluster-item {{
            flex: 0 0 180px;
            text-align: center;
            border: 1px solid #ccc;
            border-radius: 10px;
            padding: 10px;
            background-color: #f9f9f9;
            box-shadow: 2px 2px 5px rgba(0,0,0,0.1);
        }}
        .cluster-item img {{
            max-width: 160px;
            max-height: 160px;
            object-fit: contain;
            margin-bottom: 8px;
        }}
        .cluster-item .name {{
            font-size: 13px;
            font-family: Arial, sans-serif;
            color: #333;
        }}
    </style>

    <div class='cluster-header'>{cluster_name} — {cluster_size} item(s)</div>
    <div class='cluster-container'>
    """

    for item in items:
        img = item["image"] or ""
        name = item["name"] or ""
        html += f"""
            <div class='cluster-item'>
                <img src="{img}" alt="Product image">
                <div class='name'>{name}</div>
            </div>
        """

    html += "</div>"
    display(HTML(html))


In [128]:
# 1. Show first 3 clusters (excluding noise)
for cid in sorted(k for k in clusters_flat if k != -1)[:3]:
    show_cluster(cid)

# 2. Show noise cluster (-1)
show_cluster(-1)

# 3. Show cluster 0 if it exists
if 0 in clusters_flat:
    show_cluster(0)

# 4. Show largest cluster
largest_cluster = max(clusters_flat.items(), key=lambda x: len(x[1]))[0]
show_cluster(largest_cluster)

# 5. Show all clusters that have exactly 2 items
for cid, items in clusters_flat.items():
    if len(items) == 2:
        show_cluster(cid)


### Manual inspection shows us that this algorithm gives still a lot of noise (Singletons) therefore again, may not be meaningful - despite the good metrics

## Another attempt at using group-by operations - but this time excluding the name_engb field as a group-key to see if we can get a more "loose" inital grouping

In [131]:
# === Step 1: Load metadata ===
df_data = pd.read_json("sneakers_sample_19062025.txt", lines=False)

# Flatten fields for grouping and processing
df_data["brand"] = df_data["data"].apply(lambda x: x.get("brand") if isinstance(x, dict) else "")
df_data["color"] = df_data["data"].apply(lambda x: x.get("color") if isinstance(x, dict) else "")
df_data["category"] = df_data["data"].apply(lambda x: x.get("category") if isinstance(x, dict) else "")
df_data["name [en_GB]"] = df_data["data"].apply(lambda x: x.get("name [en_GB]") if isinstance(x, dict) else "")
df_data["shortDescription [en_GB]"] = df_data["data"].apply(lambda x: x.get("shortDescription") if isinstance(x, dict) else "")

# === Step 2: Group by metadata keys (excluding image info) ===
group_keys = ['brand', 'category']
group_sizes = df_data.groupby(group_keys).size().reset_index(name='count')
duplicates = group_sizes[group_sizes["count"] > 1]
identical_groups = df_data.merge(duplicates[group_keys], on=group_keys, how="inner")
identical_groups["product_group_id"] = identical_groups.groupby(group_keys).ngroup()

# === Step 3: Prepare sentence transformer ===
text_model = SentenceTransformer("all-MiniLM-L6-v2")

# === Step 4: Cluster text-only features per group ===
results_all = {}

for group_id, group_df in identical_groups.groupby("product_group_id"):
    # Extract text
    names = group_df["data"].apply(lambda x: x.get("name [en_GB]", "") if isinstance(x, dict) else "")
    brands = group_df["data"].apply(lambda x: x.get("brand", "") if isinstance(x, dict) else "")
    colors = group_df["data"].apply(lambda x: x.get("color", "") if isinstance(x, dict) else "")
    descriptions = group_df["data"].apply(lambda x: x.get("shortDescription", "") if isinstance(x, dict) else "")
    texts = (names + " " + brands + " " + colors + " " + descriptions).fillna("").tolist()
    text_vecs = text_model.encode(texts, normalize_embeddings=True)

    # Cluster text embeddings
    clusterer = hdbscan.HDBSCAN(min_cluster_size=2, metric='euclidean')
    labels = clusterer.fit_predict(text_vecs)
    group_df = group_df.copy()
    group_df["cluster_id"] = labels

    # Save clusters as JSON
    grouped_json = defaultdict(list)
    for _, row in group_df.iterrows():
        grouped_json[int(row["cluster_id"])].append({
            "product_id": row.get("mirakl_product_id"),
            "name": row.get("data", {}).get("name [en_GB]", ""),
            "image": row.get("data", {}).get("mainImage", {}).get("original_url")
        })

    results_all[f"group_{group_id}"] = grouped_json

# === Step 5: Save results ===
with open("text_only_clustered_groups_3.json", "w") as f:
    json.dump(results_all, f, indent=2)

print("Text-only clustering complete!")

Text-only clustering complete!


In [132]:
# === Step 6: Visualize results ===
with open("text_only_clustered_groups_3.json", "r") as f:
    clusters_by_group = json.load(f)

# === Example usage ===
show_group_cluster(0)
show_group_cluster(1, cluster_id=0)

## Evaluating

In [134]:
## this code assigns the cluster id and saves the new dataframe to be loaded later

# Load the JSON file
with open("text_only_clustered_groups_3.json", "r") as f:
    results_all = json.load(f)

# Flatten into a list of product_id and predicted cluster label
records = []
for group_key, clusters in results_all.items():
    for cluster_id, items in clusters.items():
        for item in items:
            records.append({
                "mirakl_product_id": item["product_id"],
                "method_metadata_grouped_3": f"{group_key}_cluster{cluster_id}"
            })

# Create and save the DataFrame
df_predicted = pd.DataFrame(records)
df_predicted.to_csv("df_with_method_metadata_grouped_3.csv", index=False)
print("Saved: df_with_method_metadata_grouped_3.csv")


Saved: df_with_method_metadata_grouped_3.csv


In [135]:

# Load predictions and ground truth
df_predicted = pd.read_csv("df_with_method_metadata_grouped_3.csv")
df_gt = pd.read_csv("sample.csv")

# Merge and clean
df = df_predicted.merge(df_gt[["mirakl_product_id", "manual_group_id"]], on="mirakl_product_id", how="inner")
df = df.dropna(subset=["method_metadata_grouped_3", "manual_group_id"])
df["manual_group_id"] = df["manual_group_id"].astype(int)

# Compute cluster purity
purity = compute_cluster_purity(df, "method_metadata_grouped_3", "manual_group_id")

# Assign predicted label from majority vote per cluster
majority_map = (
    df.groupby("method_metadata_grouped_3")["manual_group_id"]
    .agg(lambda x: x.value_counts().idxmax())
    .to_dict()
)
df["predicted_manual_group_id"] = df["method_metadata_grouped_3"].map(majority_map)
df["predicted_manual_group_id"] = df["predicted_manual_group_id"].astype(int)

# Compute pairwise confusion matrix
conf_matrix = pair_confusion_matrix(df["manual_group_id"], df["predicted_manual_group_id"])
TP = conf_matrix[1, 1]
FP = conf_matrix[0, 1]
FN = conf_matrix[1, 0]
TN = conf_matrix[0, 0]

# Micro-averaged metrics
micro_precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
micro_accuracy = (TP + TN) / (TP + FP + FN + TN)

# Report
print(f"method_metadata_grouped_3 Cluster Purity: {purity:.4f}")
print(f"method_metadata_grouped_3 Micro-average Precision (Pairwise): {micro_precision:.4f}")
print(f"method_metadata_grouped_3 Micro-average Accuracy (Pairwise): {micro_accuracy:.4f}")


method_metadata_grouped_3 Cluster Purity: 0.6416
method_metadata_grouped_3 Micro-average Precision (Pairwise): 0.1391
method_metadata_grouped_3 Micro-average Accuracy (Pairwise): 0.9924


## If we had better results on the above method, we could proceed with using color as a group-key as well in order to attempt to detect variants (a task which we could not evaluate as our sample is manually labelled with the groups of variants - but since this method does not perform as well as the above hybrid methods - which utilise image embeddings as well - then we shall infer that combining image and textual metadata results in the most efficient clustering for this specific task.

## Summary:

In this notebook, we explore the use of contextual metadata embeddings to enhance clustering for identifying product variants and potential duplicates. Starting from traditional unsupervised methods, we incrementally introduce grouping logic and representation learning techniques to improve performance. The first method investigates the impact of replacing traditional TF-IDF embeddings with contextual embeddings generated from a SentenceTransformer model, while keeping image embeddings fixed. These fused embeddings (image + contextual text) are clustered using HDBSCAN, a density-based algorithm that can handle variable cluster sizes and noise. Manual visual inspection reveals this baseline method does not produce tight, consistent clusters — leading us to conclude that a stricter grouping strategy is necessary.

These leads us to method *, where we introduce a manual grouping step using groupby on brand and category, prior to embedding. This grouping restricts the clustering to only occur within semantically related subsets of the data, thus improving intra-group cohesion. Within each group, contextual metadata embeddings and image embeddings are combined as before and clustered using HDBSCAN. This approach shows stronger alignment with the expected groupings, confirmed both visually and quantitatively. Still, we observe cluster impurity in some cases, prompting further tightening of the grouping criteria. In method **, we enhance the group key to include name_en_gb (product name) alongside brand and category. This refinement yields the most precise clustering performance, with nearly perfect scores:
→ Cluster Purity: 0.9965
→ Micro-average Precision: 0.9700
→ Micro-average Accuracy: 0.99995
These metrics, evaluated against a manually labeled variant dataset, demonstrate that a hybrid approach combining contextual embeddings, image embeddings, and structured grouping yields the most effective results.

Method *** shifts the objective to identifying identical products (not just variants) by adding color to the grouping key. This imposes a much stricter definition of similarity, assuming that the color feature is indeed correct and representative of the true color of the product and is the factor that determines a variant from an identical sneaker. Due to the lack of ground truth for truly identical products in the dataset, this method could not be evaluated quantitatively, but remains conceptually useful for deduplication. 

In method ****, the approach from method ** is retained, but input images are converted to grayscale before embedding. This is intended to eliminate color bias from the visual embeddings, ensuring that clustering decisions aren't overly driven by superficial color differences. While this grayscale variant performs robustly, it does not outperform method **, suggesting that color contributes positively in distinguishing product variants.

Finally, we test whether image embeddings are necessary at all. By excluding image features and relying solely on grouped contextual embeddings, we find that performance drops, especially when groupings become too strict or embeddings too sparse. These results reaffirm that a hybrid strategy, combining manual grouping, contextual language models, and visual features, leads to the most meaningful and high-quality product clustering.